# Langchain - Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

Add middleware by passing them to `create_agent`.

In [1]:
import os
from dotenv import load_dotenv

"""                     Setup.                       """
# Load environment variables from .env file.
load_dotenv()
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

    agent_with_middleware = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[...],
        middleware=[
            SummarizationMiddleware(...),
            HumanInTheLoopMiddleware(...)
        ],
    )
    ```

- The agent loop
    
    The core agent loop involves calling a model, letting it choose tools to execute, and then finishing when it calls no more tools.

    <div>
    <img src="../../docs/i-tutorials/ii-lc-agent-loop-01.png" width="400"/>
    </div>

    Middleware exposes hooks before and after each of those steps:

    <div>
    <img src="../../docs/i-tutorials/ii-lc-agent-loop-02.png" width="600"/>
    </div>

## 1. Prebuilt (Built-in) middleware

LangChain and Deep Agents provide prebuilt middleware for common use cases. Each middleware is production-ready and configurable for your specific needs.
​
### 1.1. Provider-agnostic middleware

The following middleware work with any LLM provider:

| Middleware        | Description                                                                    |
|:------------------|:-------------------------------------------------------------------------------|
| __Summarization__     | Automatically summarize conversation history when approaching token limits.    |
| __Human-in-the-loop__ | Pause execution for human approval of tool calls.                              |
| __Model call limit__  | Limit the number of model calls to prevent excessive costs.                    |
| __Tool call limit__   | Control tool execution by limiting call counts.                                |
| __Model fallback__    | Automatically fallback to alternative models when primary fails.               |
| __PII detection__     | Detect and handle Personally Identifiable Information (PII).                   |
| __To-do list__        | Equip agents with task planning and tracking capabilities.                     |
| __LLM tool selector__ | Use an LLM to select relevant tools before calling main model.                 |
| __Tool retry__        | Automatically retry failed tool calls with exponential backoff.                |
| __Model retry__       | Automatically retry failed model calls with exponential backoff.               |
| __LLM tool emulator__ | Emulate tool execution using an LLM for testing purposes.                      |
| __Context editing__   | Manage conversation context by trimming or clearing tool uses.                 |
| __Shell tool__        | Expose a persistent shell session to agents for command execution.             |
| __File search__       | Provide Glob and Grep search tools over filesystem files.                      |
| __Filesystem__        | Provide agents with a filesystem for storing context and long-term memories.   |
| __Subagent__          | Add the ability to spawn subagents.                                            |

#### 1.1.1. Summarization

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware?_gl=1*swfdsd*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4MTUyMDckbzIwJGcxJHQxNzc0ODE1MzkzJGo2MCRsMCRoMA..">`SummarizationMiddleware`</a>

Configuration options available are:
- `model` (string | `BaseChatModel` _required_):

    Model for generating summaries. Can be a model identifier string (e.g., `'openai:gpt-4.1-mini'`) or a `BaseChatModel` instance.
​
- `trigger` (ContextSize | list[ContextSize] | None):

    Condition(s) for triggering summarization. Can be:
    - A single `ContextSize` tuple (specified condition must be met)
    - A list of `ContextSize` tuples (any condition must be met - OR logic)

    Condition should be one of the following:
    - `fraction` (float): Fraction of model’s context size (0-1)
    - `tokens` (int): Absolute token count
    - `messages` (int): Message count

    At least one condition must be specified. If not provided, summarization will not trigger automatically.
    
    See the API reference for <a href="https://reference.langchain.com/python/langchain/agents/middleware/summarization/ContextSize?_gl=1*49fcgx*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjc2OTQkbzI0JGcwJHQxNzc1MDY3Njk0JGo2MCRsMCRoMA..">ContextSize</a> for more information.
​
- `keep` (ContextSize `default:"('messages', 20)"`):

    How much context to preserve after summarization. Specify exactly one of:
    - `fraction` (float): Fraction of model’s context size to keep (0-1)
    - `tokens` (int): Absolute token count to keep
    - `messages` (int): Number of recent messages to keep
​
- `token_counter` (function):

    Custom token counting function. Defaults to character-based counting.
​
- `summary_prompt` (string):

    Custom prompt template for summarization. Uses built-in template if not specified. The template should include `{messages}` placeholder where conversation history will be inserted.
​
- `trim_tokens_to_summarize` (number `default:"4000"`):

    Maximum number of tokens to include when generating the summary. Messages will be trimmed to fit this limit before summarization.

The summarization middleware monitors message token counts and automatically summarizes older messages when thresholds are reached.

__Trigger conditions__ control when summarization runs:
- Single condition object (specified must be met)
- Array of conditions (any condition must be met - OR logic)
- Each condition can use `fraction` (of model’s context size), `tokens` (absolute count), or `messages` (message count)

__Keep condition__ control how much context to preserve (specify exactly one):
- `fraction` - Fraction of model’s context size to keep
- `tokens` - Absolute token count to keep
- `messages` - Number of recent messages to keep

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import SummarizationMiddleware

    # Single condition: trigger if tokens >= 4000
    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[your_weather_tool, your_calculator_tool],
        middleware=[
            SummarizationMiddleware(
                model="google_genai:gemini-2.5-flash"
                trigger=("tokens", 4000),
                keep=("messages", 20),
            ),
        ],
    )

    # Multiple conditions: trigger if number of tokens >= 3000 OR messages >= 6
    agent2 = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[your_weather_tool, your_calculator_tool],
        middleware=[
            SummarizationMiddleware(
                model="google_genai:gemini-2.5-flash"
                trigger=[
                    ("tokens", 3000),
                    ("messages", 6),
                ],
                keep=("messages", 20),
            ),
        ],
    )

    # Using fractional limits
    agent3 = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[your_weather_tool, your_calculator_tool],
        middleware=[
            SummarizationMiddleware(
                model="google_genai:gemini-2.5-flash"
                trigger=("fraction", 0.8),
                keep=("fraction", 0.3),
            ),
        ],
    )
    ```

#### 1.1.2. Human-in-the-loop

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HumanInTheLoopMiddleware?_gl=1*1du9g9g*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4MTUyMDckbzIwJGcxJHQxNzc0ODE2MzE5JGo2MCRsMCRoMA..">`HumanInTheLoopMiddleware`</a>

For more details and advanced usage, check <a href="https://docs.langchain.com/oss/python/langchain/human-in-the-loop">here</a>.

<div class="alert alert-block alert-warning">

ℹ️ Human-in-the-loop middleware requires a 
<a href="https://docs.langchain.com/oss/python/langgraph/persistence#checkpoints">checkpointer</a> to maintain state across interruptions.

</div>

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import HumanInTheLoopMiddleware
    from langgraph.checkpoint.memory import InMemorySaver

    def your_read_email_tool(email_id: str) -> str:
        """Mock function to read an email by its ID."""
        return f"Email content for ID: {email_id}"

    def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
        """Mock function to send an email."""
        return f"Email sent to {recipient} with subject '{subject}'"

    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[your_read_email_tool, your_send_email_tool],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "your_send_email_tool": {
                        "allowed_decisions": ["approve", "edit", "reject"],
                    },
                    "your_read_email_tool": False,
                }
            ),
        ],
    )
    ```

#### 1.1.3. Model call limit

Limit the number of model calls to prevent infinite loops or excessive costs. Model call limit is useful for the following:
- Preventing runaway agents from making too many API calls.
- Enforcing cost controls on production deployments.
- Testing agent behavior within specific call budgets.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/model_call_limit/ModelCallLimitMiddleware?_gl=1*4kz6o9*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4OTA0MjYkbzIxJGcxJHQxNzc0ODkwNzcwJGoxNiRsMCRoMA..">`ModelCallLimitMiddleware`</a>

Configuration options available are:
- `thread_limit` (number):
    
    Maximum model calls across all runs in a thread. Defaults to no limit.
​
- `run_limit` (number):
    
    Maximum model calls per single invocation. Defaults to no limit.
​
- `exit_behavior` (string `default`:"end"):
    
    Behavior when limit is reached. Options: `'end'` (graceful termination) or `'error'` (raise exception).

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ModelCallLimitMiddleware
    from langgraph.checkpoint.memory import InMemorySaver

    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        checkpointer=InMemorySaver(),  # required for thread limiting
        tools=[],
        middleware=[
            ModelCallLimitMiddleware(
                thread_limit=10,
                run_limit=5,
                exit_behavior="end",
            ),
        ],
    )
```

#### 1.1.4. Tool call limit

Control agent execution by limiting the number of tool calls, either globally across all tools or for specific tools. Tool call limits are useful for the following:
- Preventing excessive calls to expensive external APIs.
- Limiting web searches or database queries.
- Enforcing rate limits on specific tool usage.
- Protecting against runaway agent loops.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/tool_call_limit/ToolCallLimitMiddleware?_gl=1*1dcx0u5*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4OTA0MjYkbzIxJGcxJHQxNzc0ODkwNzcwJGoxNiRsMCRoMA..">`ToolCallLimitMiddleware`</a>

Configuration options available are:
- `tool_name` (string):
    
    Name of specific tool to limit. If not provided, limits apply to all tools globally.
​
- `thread_limit` (number):
    
    Maximum tool calls across all runs in a thread (conversation). Persists across multiple invocations with the same thread ID. Requires a checkpointer to maintain state. `None` means no thread limit.
​
- `run_limit` (number):
    
    Maximum tool calls per single invocation (one user message → response cycle). Resets with each new user message. `None` means no run limit.
    
    __Note:__ At least one of `thread_limit` or `run_limit` must be specified.
​
- `exit_behavior` (string `default:"continue"`):
    
    Behavior when limit is reached:

    - `'continue'` (default) - 
        
        Block exceeded tool calls with error messages, let other tools and the model continue. The model decides when to end based on the error messages.

    - `'error'` - 
        
        Raise a `ToolCallLimitExceededError` exception, stopping execution immediately

    - `'end'` - 
        
        Stop execution immediately with a `ToolMessage` and AI message for the exceeded tool call. Only works when limiting a single tool; raises `NotImplementedError` if other tools have pending calls.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ToolCallLimitMiddleware


    global_limiter = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)
    search_limiter = ToolCallLimitMiddleware(tool_name="search", thread_limit=5, run_limit=3)
    database_limiter = ToolCallLimitMiddleware(tool_name="query_database", thread_limit=10)
    strict_limiter = ToolCallLimitMiddleware(tool_name="scrape_webpage", run_limit=2, exit_behavior="error")

    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[search_tool, database_tool, scraper_tool],
        middleware=[global_limiter, search_limiter, database_limiter, strict_limiter],
    )
    ```

#### 1.1.5. Model fallback

Automatically fallback to alternative models when the primary model fails. Model fallback is useful for the following:
- Building resilient agents that handle model outages.
- Cost optimization by falling back to cheaper models.
- Provider redundancy across OpenAI, Anthropic, etc.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/model_fallback/ModelFallbackMiddleware?_gl=1*1m9ape*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4OTA0MjYkbzIxJGcxJHQxNzc0ODkwNzcwJGoxNiRsMCRoMA..">`ModelFallbackMiddleware`</a>

Configuration options available are:
- `first_model` (string | `BaseChatModel` _required_):
    
    First fallback model to try when the primary model fails. Can be a model identifier string (e.g., `'openai:gpt-4.1-mini'`) or a `BaseChatModel` instance.
​
- `*additional_models` (string | `BaseChatModel`):
    
    Additional fallback models to try in order if previous models fail

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ModelFallbackMiddleware

    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[],
        middleware=[
            ModelFallbackMiddleware(
                "gpt-4.1-mini",
                "claude-3-5-sonnet-20241022",
            ),
        ],
    )
    ```

#### 1.1.6. PII detection

Detect and handle Personally Identifiable Information (PII) in conversations using configurable strategies. PII detection is useful for the following:
- Healthcare and financial applications with compliance requirements.
- Customer service agents that need to sanitize logs.
- Any application handling sensitive user data.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware?_gl=1*9tfk27*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzQ4OTA0MjYkbzIxJGcxJHQxNzc0ODkwNzcwJGoxNiRsMCRoMA..">`PIIMiddleware`</a>

Check <a href="https://docs.langchain.com/oss/python/langchain/middleware/built-in#custom-pii-types">`Custom PII types`</a> for more info.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import PIIMiddleware

    agent = create_agent(
        model="gpt-4.1",
        tools=[],
        middleware=[
            PIIMiddleware("email", strategy="redact", apply_to_input=True),
            PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        ],
    )
    ```

#### 1.1.7. To-do list

Equip agents with task planning and tracking capabilities for complex multi-step tasks. To-do lists are useful for the following:
- Complex multi-step tasks requiring coordination across multiple tools.
- Long-running operations where progress visibility is important.

<div class="alert alert-block alert-info">

ℹ️ This middleware automatically provides agents with a `write_todos` tool and system prompts to guide effective task planning.

</div>

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/todo/TodoListMiddleware?_gl=1*3kj42p*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`TodoListMiddleware`</a>

Configuration options available are:
- `system_prompt` (string):

    Custom system prompt for guiding todo usage. Uses built-in prompt if not specified.
- `tool_description` (string):

    Custom description for the `write_todos` tool. Uses built-in description if not specified.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import TodoListMiddleware

    agent = create_agent(
        model="gpt-4.1",
        tools=[read_file, write_file, run_tests],
        middleware=[TodoListMiddleware()],
    )
    ```

#### 1.1.8. LLM tool selector

Use an LLM to intelligently select relevant tools before calling the main model. LLM tool selectors are useful for the following:
- Agents with many tools (10+) where most aren’t relevant per query.
- Reducing token usage by filtering irrelevant tools.
- Improving model focus and accuracy.

This middleware uses structured output to ask an LLM which tools are most relevant for the current query. The structured output schema defines the available tool names and descriptions. Model providers often add this structured output information to the system prompt behind the scenes.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/tool_selection/LLMToolSelectorMiddleware?_gl=1*1dpekcg*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`LLMToolSelectorMiddleware`</a>

Configuration options available are:
- `model` (string | `BaseChatModel`):

    Model for tool selection. Can be a model identifier string (e.g., `'openai:gpt-4.1-mini'`) or a BaseChatModel instance.
    
    Defaults to the agent’s main model.
​
- `system_prompt` (string):
    
    Instructions for the selection model. Uses built-in prompt if not specified.
​
- `max_tools` (number):

    Maximum number of tools to select. If the model selects more, only the first max_tools will be used. No limit if not specified.
​
- `always_include` (list[string]):

    Tool names to always include regardless of selection. These do not count against the max_tools limit.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import LLMToolSelectorMiddleware

    agent = create_agent(
        model="groq:llama-3.3-70b-versatile",
        tools=[tool1, tool2, tool3, tool4, tool5, ...],
        middleware=[
            LLMToolSelectorMiddleware(
                model="google_genai:gemini-2.5-flash",
                max_tools=3,
                always_include=["search"],
            ),
        ],
    )
    ```

#### 1.1.9. Tool retry

Automatically retry failed tool calls with configurable exponential backoff. Tool retry is useful for the following:
- Handling transient failures in external API calls.
- Improving reliability of network-dependent tools.
- Building resilient agents that gracefully handle temporary errors.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/tool_retry/ToolRetryMiddleware?_gl=1*1o7gafb*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`ToolRetryMiddleware`</a>


Configuration options available are:
- `max_retries` (number `default:"2"`):

    Maximum number of retry attempts after the initial call (3 total attempts with default)
​
- `tools` (list[`BaseTool` | str]):

    Optional list of tools or tool names to apply retry logic to. If None, applies to all tools.
​
- `retry_on` (tuple[type[Exception], ...] | callable `default:"(Exception,)"`):

    Either a tuple of exception types to retry on, or a callable that takes an exception and returns True if it should be retried.
​
- `on_failure` (string | callable `default:"return_message"`):

    Behavior when all retries are exhausted. Options:

    - `'return_message'` - Return a `ToolMessage` with error details (allows LLM to handle failure)
    - `'raise'` - Re-raise the exception (stops agent execution)
    - Custom callable - Function that takes the exception and returns a string for the `ToolMessage` content
​
- `backoff_factor` (number `default:"2.0"`):

    Multiplier for exponential backoff. Each retry waits `initial_delay * (backoff_factor ** retry_number)` seconds. Set to `0.0` for constant delay.
​
- `initial_delay` (number `default:"1.0"`):

    Initial delay in seconds before first retry
​
- `max_delay` (number `default:"60.0"`):

    Maximum delay in seconds between retries (caps exponential backoff growth)
​
- `jitter` (boolean `default:"true"`):

    Whether to add random jitter (±25%) to delay to avoid thundering herd

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ToolRetryMiddleware

    agent = create_agent(
        model="gpt-4.1",
        tools=[search_tool, database_tool, api_tool],
        middleware=[
            ToolRetryMiddleware(
                max_retries=3,
                backoff_factor=2.0,
                initial_delay=1.0,
                max_delay=60.0,
                jitter=True,
                tools=["api_tool"],
                retry_on=(ConnectionError, TimeoutError),
                on_failure="continue",
            ),
        ],
    )
    ```

#### 1.1.10. Model retry

Automatically retry failed model calls with configurable exponential backoff. Model retry is useful for the following:
- Handling transient failures in model API calls.
- Improving reliability of network-dependent model requests.
- Building resilient agents that gracefully handle temporary model errors.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/model_retry/ModelRetryMiddleware?_gl=1*1sirrqh*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`ModelRetryMiddleware`</a>


Configuration options available are:
- `max_retries` (number `default:"2"`):

    Maximum number of retry attempts after the initial call (3 total attempts with default)
​
- retry_on
(tuple[type[Exception], ...] | callable `default:"(Exception,)"`):

    Either a tuple of exception types to retry on, or a callable that takes an exception and returns `True` if it should be retried.
​
- `on_failure` (string | callable `default:"continue"`):

    Behavior when all retries are exhausted. Options:
    
    - `'continue'` (default) - Return an `AIMessage` with error details, allowing the agent to potentially handle the failure gracefully
    - `'error'` - Re-raise the exception (stops agent execution)
    - Custom callable - Function that takes the exception and returns a string for the `AIMessage` content
​
- `backoff_factor` (number `default:"2.0"`):

    Multiplier for exponential backoff. Each retry waits `initial_delay * (backoff_factor ** retry_number)` seconds. Set to `0.0` for constant delay.
​
- `initial_delay` (number `default:"1.0"`):

    Initial delay in seconds before first retry
​
- `max_delay` (number `default:"60.0"`):

    Maximum delay in seconds between retries (caps exponential backoff growth)
​
- `jitter` (boolean `default:"true"`):

    Whether to add random jitter (`±25%`) to delay to avoid thundering herd

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ModelRetryMiddleware

    agent = create_agent(
        model="gpt-4.1",
        tools=[search_tool, database_tool],
        middleware=[
            ModelRetryMiddleware(
                max_retries=3,
                backoff_factor=2.0,
                initial_delay=1.0,
            ),
        ],
    )
    ```

#### 1.1.11. LLM tool emulator

Emulate tool execution using an LLM for testing purposes, replacing actual tool calls with AI-generated responses. LLM tool emulators are useful for the following:
- Testing agent behavior without executing real tools.
- Developing agents when external tools are unavailable or expensive.
- Prototyping agent workflows before implementing actual tools.

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/tool_emulator/LLMToolEmulator?_gl=1*15me55h*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`LLMToolEmulator`</a>


Configuration options available are:
- `tools` (list[str | `BaseTool`]):

    List of tool names (str) or BaseTool instances to emulate. If `None` (default), ALL tools will be emulated. If empty list `[]`, no tools will be emulated. If array with tool names/instances, only those tools will be emulated.
​
- `model` (string | `BaseChatModel`):

    Model to use for generating emulated tool responses. Can be a model identifier string (e.g., `'anthropic:claude-sonnet-4-6'`) or a `BaseChatModel` instance. Defaults to the agent’s model if not specified.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import LLMToolEmulator
    from langchain.tools import tool

    @tool
    def get_weather(location: str) -> str:
        """Get the current weather for a location."""
        return f"Weather in {location}"

    @tool
    def send_email(to: str, subject: str, body: str) -> str:
        """Send an email."""
        return "Email sent"

    # Emulate all tools (default behavior)
    agent = create_agent(
        model="gpt-4.1",
        tools=[get_weather, send_email],
        middleware=[LLMToolEmulator()],
    )

    # Emulate specific tools only
    agent2 = create_agent(
        model="gpt-4.1",
        tools=[get_weather, send_email],
        middleware=[LLMToolEmulator(tools=["get_weather"])],
    )

    # Use custom model for emulation
    agent4 = create_agent(
        model="gpt-4.1",
        tools=[get_weather, send_email],
        middleware=[LLMToolEmulator(model="claude-sonnet-4-6")],
    )
    ```

#### 1.1.12. Context editing

Manage conversation context by clearing older tool call outputs when token limits are reached, while preserving recent results. This helps keep context windows manageable in long conversations with many tool calls. Context editing is useful for the following:
- Long conversations with many tool calls that exceed token limits
- Reducing token costs by removing older tool outputs that are no longer relevant
- Maintaining only the most recent N tool results in context

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ContextEditingMiddleware?_gl=1*jwvolg*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`ContextEditingMiddleware`</a>, <a href="https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ClearToolUsesEdit?_gl=1*jwvolg*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjM5NjQkbzIzJGcxJHQxNzc1MDY0MjI5JGo2MCRsMCRoMA..">`ClearToolUsesEdit`</a>

Configuration options available for `ContextEditingMiddleware` are:
- `edits` (list[ContextEdit] `default:"[ClearToolUsesEdit()]"`):

    List of `ContextEdit` strategies to apply
​
- `token_count_method` (string `default:"approximate"`):

    Token counting method. Options: `'approximate'` or `'model'`

Configuration options available for `ClearToolUsesEdit` are:

- `trigger` (number `default:"100000"`):

    Token count that triggers the edit. When the conversation exceeds this token count, older tool outputs will be cleared.
​
- `clear_at_least` (number `default:"0"`):

    Minimum number of tokens to reclaim when the edit runs. If set to 0, clears as much as needed.
​
- `keep` (number `default:"3"`):

    Number of most recent tool results that must be preserved. These will never be cleared.
​
- `clear_tool_inputs` (boolean `default:"False"`):

    Whether to clear the originating tool call parameters on the AI message. When `True`, tool call arguments are replaced with empty objects.
​
- `exclude_tools` (list[string] `default:"()"`):

    List of tool names to exclude from clearing. These tools will never have their outputs cleared.
​
- `placeholder` (string `default:"[cleared]"`):

    Placeholder text inserted for cleared tool outputs. This replaces the original tool message content.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

    agent = create_agent(
        model="gpt-4.1",
        tools=[search_tool, your_calculator_tool, database_tool],
        middleware=[
            ContextEditingMiddleware(
                edits=[
                    ClearToolUsesEdit(
                        trigger=2000,
                        keep=3,
                        clear_tool_inputs=False,
                        exclude_tools=[],
                        placeholder="[cleared]",
                    ),
                ],
            ),
        ],
    )
    ```

#### 1.1.13. Shell tool

Expose a persistent shell session to agents for command execution. Shell tool middleware is useful for the following:
- Agents that need to execute system commands
- Development and deployment automation tasks
- Testing and validation workflows
- File system operations and script execution

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/shell_tool/ShellToolMiddleware?_gl=1*1bzv86q*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjc2OTQkbzI0JGcwJHQxNzc1MDY3Njk0JGo2MCRsMCRoMA..">`ShellToolMiddleware`</a>

<div class="alert alert-block alert-warning">

⚠️ Security consideration: Use appropriate execution policies (`HostExecutionPolicy`, `DockerExecutionPolicy`, or `CodexSandboxExecutionPolicy`) to match your deployment’s security requirements.

</div>

Configuration options available are:
- `workspace_root` (str | Path | None):

    Base directory for the shell session. If omitted, a temporary directory is created when the agent starts and removed when it ends.
​
- `startup_commands` (tuple[str, ...] | list[str] | str | None):

    Optional commands executed sequentially after the session starts
​
- `shutdown_commands` (tuple[str, ...] | list[str] | str | None):

    Optional commands executed before the session shuts down
​
- `execution_policy` (BaseExecutionPolicy | None):

    Execution policy controlling timeouts, output limits, and resource configuration. Options:

    - `HostExecutionPolicy` - Full host access (default); best for trusted environments where the agent already runs inside a container or VM
    - `DockerExecutionPolicy` - Launches a separate Docker container for each agent run, providing harder isolation
    - `CodexSandboxExecutionPolicy` - Reuses the Codex CLI sandbox for additional syscall/filesystem restrictions
​
- `redaction_rules` (tuple[RedactionRule, ...] | list[RedactionRule] | None):

    Optional redaction rules to sanitize command output before returning it to the model.

    <div class="alert alert-block alert-warning">

    ⚠️ Redaction rules are applied post execution and do not prevent exfiltration of secrets or sensitive data when using `HostExecutionPolicy`.

    </div>
​
- `tool_description` (str | None):

    Optional override for the registered shell tool description
​
- `shell_command` (Sequence[str] | str | None):

    Optional shell executable (string) or argument sequence used to launch the persistent session. Defaults to `/bin/bash`.
​
- `env` (Mapping[str, Any] | None):

    Optional environment variables to supply to the shell session. Values are coerced to strings before command execution.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import (
        ShellToolMiddleware,
        HostExecutionPolicy,
        DockerExecutionPolicy,
        RedactionRule,
    )

    # Basic shell tool with host execution
    agent = create_agent(
        model="gpt-4.1",
        tools=[search_tool],
        middleware=[
            ShellToolMiddleware(
                workspace_root="/workspace",
                execution_policy=HostExecutionPolicy(),
            ),
        ],
    )

    # Docker isolation with startup commands
    agent_docker = create_agent(
        model="gpt-4.1",
        tools=[],
        middleware=[
            ShellToolMiddleware(
                workspace_root="/workspace",
                startup_commands=["pip install requests", "export PYTHONPATH=/workspace"],
                execution_policy=DockerExecutionPolicy(
                    image="python:3.11-slim",
                    command_timeout=60.0,
                ),
            ),
        ],
    )

    # With output redaction (applied post execution)
    agent_redacted = create_agent(
        model="gpt-4.1",
        tools=[],
        middleware=[
            ShellToolMiddleware(
                workspace_root="/workspace",
                redaction_rules=[
                    RedactionRule(pii_type="api_key", detector=r"sk-[a-zA-Z0-9]{32}"),
                ],
            ),
        ],
    )
    ```

#### 1.1.14. File search

Provide Glob and Grep search tools over a filesystem. File search middleware is useful for the following:
- Code exploration and analysis
- Finding files by name patterns
- Searching code content with regex
- Large codebases where file discovery is needed

__API reference:__ <a href="https://reference.langchain.com/python/langchain/agents/middleware/file_search/FilesystemFileSearchMiddleware?_gl=1*1co5vdm*_gcl_au*NTQwNTUwMTYwLjE3NzQwOTY2ODA.*_ga*MTYxNTQzNTA3NC4xNzc0MDk2Njgw*_ga_47WX3HKKY2*czE3NzUwNjc2OTQkbzI0JGcwJHQxNzc1MDY3Njk0JGo2MCRsMCRoMA..">`FilesystemFileSearchMiddleware`</a>

Configuration options available are:
- `root_path` (str _required_):

    Root directory to search. All file operations are relative to this path.

- `use_ripgrep` (bool `default:"True"`):

    Whether to use ripgrep for search. Falls back to Python regex if ripgrep is unavailable.

- `max_file_size_mb` (int `default:"10"`):

    Maximum file size to search in MB. Files larger than this are skipped.

The middleware adds two search tools to agents:
- __Glob tool__ - Fast file pattern matching:

    - Supports patterns like `**/*.py`, `src/**/*.ts`
    - Returns matching file paths sorted by modification time

- __Grep tool__ - Content search with regex:

    - Full regex syntax support
    - Filter by file patterns with `include` parameter
    - Three output modes: `files_with_matches`, `content`, `count`

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.agents.middleware import FilesystemFileSearchMiddleware
    from langchain.messages import HumanMessage

    agent = create_agent(
        model="gpt-4.1",
        tools=[],
        middleware=[
            FilesystemFileSearchMiddleware(
                root_path="/workspace",
                use_ripgrep=True,
                max_file_size_mb=10,
            ),
        ],
    )

    # Agent can now use glob_search and grep_search tools
    result = agent.invoke({
        "messages": [HumanMessage("Find all Python files containing 'async def'")]
    })

    # The agent will use:
    # 1. glob_search(pattern="**/*.py") to find Python files
    # 2. grep_search(pattern="async def", include="*.py") to find async functions
    ```

#### 1.1.15. Filesystem

Context engineering is a main challenge in building effective agents. This is particularly difficult when using tools that return variable-length results (for example, `web_search` and RAG), as long tool results can quickly fill your context window.

`FilesystemMiddleware` from <a href="https://docs.langchain.com/oss/python/deepagents/overview">Deep Agents</a> provides four tools for interacting with both short-term and long-term memory:
- `ls`: List the files in the filesystem
- `read_file`: Read an entire file or a certain number of lines from a file
- `write_file`: Write a new file to the filesystem
- `edit_file`: Edit an existing file in the filesystem

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from deepagents.middleware.filesystem import FilesystemMiddleware

    # FilesystemMiddleware is included by default in create_deep_agent
    # You can customize it if building a custom agent
    agent = create_agent(
        model="claude-sonnet-4-6",
        middleware=[
            FilesystemMiddleware(
                backend=None,  # Optional: custom backend (defaults to StateBackend)
                system_prompt="Write to the filesystem when...",  # Optional custom addition to the system prompt
                custom_tool_descriptions={
                    "ls": "Use the ls tool when...",
                    "read_file": "Use the read_file tool to..."
                }  # Optional: Custom descriptions for filesystem tools
            ),
        ],
    )
    ```

_Short-term vs. long-term filesystem_

By default, these tools write to a local “filesystem” in your graph state. To enable persistent storage across threads, configure a `CompositeBackend` that routes specific paths (like `/memories/`) to a `StoreBackend`.

When you configure a `CompositeBackend` with a `StoreBackend` for `/memories/`, any files prefixed with __/memories/__ are saved to persistent storage and survive across different threads. Files without this prefix remain in ephemeral state storage.

_`Example:`_

- 
    ```python
    from langchain.agents import create_agent
    from deepagents.middleware import FilesystemMiddleware
    from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
    from langgraph.store.memory import InMemoryStore

    store = InMemoryStore()

    agent = create_agent(
        model="claude-sonnet-4-6",
        store=store,
        middleware=[
            FilesystemMiddleware(
                backend=lambda rt: CompositeBackend(
                    default=StateBackend(rt),
                    routes={"/memories/": StoreBackend(rt)}
                ),
                custom_tool_descriptions={
                    "ls": "Use the ls tool when...",
                    "read_file": "Use the read_file tool to..."
                }  # Optional: Custom descriptions for filesystem tools
            ),
        ],
    )
    ```

#### 1.1.16. Subagent

Handing off tasks to subagents isolates context, keeping the main (supervisor) agent’s context window clean while still going deep on a task.

The subagents middleware from _Deep Agents_ allows you to supply subagents through a task `tool`.

A subagent is defined with a `name`, `description`, `system_prompt`, and `tools`. You can also provide a subagent with a _custom model_, or with _additional middleware_. This can be particularly useful when you want to give the subagent an additional state key to share with the main agent.

_`Example:`_

- 
    ```python
    from langchain.tools import tool
    from langchain.agents import create_agent
    from deepagents.middleware.subagents import SubAgentMiddleware

    @tool
    def get_weather(city: str) -> str:
        """Get the weather in a city."""
        return f"The weather in {city} is sunny."

    agent = create_agent(
        model="claude-sonnet-4-6",
        middleware=[
            SubAgentMiddleware(
                default_model="claude-sonnet-4-6",
                default_tools=[],
                subagents=[
                    {
                        "name": "weather",
                        "description": "This subagent can get weather in cities.",
                        "system_prompt": "Use the get_weather tool to get the weather in a city.",
                        "tools": [get_weather],
                        "model": "gpt-4.1",
                        "middleware": [],
                    }
                ],
            )
        ],
    )
    ```

### 1.2. Provider-specific middleware

These middleware are optimized for specific LLM providers. See each provider’s documentation for full details and examples.
- <a href="https://docs.langchain.com/oss/python/integrations/middleware/anthropic">`Anthropic`</a> - Prompt caching, bash tool, text editor, memory, and file search middleware for Claude models.
- <a href="https://docs.langchain.com/oss/python/integrations/middleware/aws">`AWS`</a> - Prompt caching middleware for Amazon Bedrock models.
- <a href="https://docs.langchain.com/oss/python/integrations/middleware/openai">`OpenAI`</a> - Content moderation middleware for OpenAI models.

## 2. Custom middleware

Build custom middleware by implementing hooks that run at specific points in the agent execution flow.

### 2.1. Hooks

Middleware provides two styles of hooks to intercept agent execution:
- __Node-style hooks__: Run sequentially at specific execution points.
- __Wrap-style hooks__: Run around each model or tool call.

#### 2.1.1. Node-style hooks

Run sequentially at specific execution points. Use for logging, validation, and state updates.

Choose the hooks your middleware needs. You can choose between node-style hooks and wrap-style hooks.

Node-style hooks run at specific execution points. Available hooks are:

| Hook           | When it runs                              |
|:---------------|:------------------------------------------|
| `before_agent` | Before agent starts (once per invocation) |
| `before_model` | Before each model call                    |
| `after_model`  | After each model response                 |
| `after_agent`  | After agent completes (once per invocation) |



_`Example: Decorator`_

- 
    ```python
    from langchain.agents.middleware import before_model, after_model, AgentState
    from langchain.messages import AIMessage
    from langgraph.runtime import Runtime
    from typing import Any

    @before_model(can_jump_to=["end"])
    def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if len(state["messages"]) >= 50:
            return {
                "messages": [AIMessage("Conversation limit reached.")],
                "jump_to": "end"
            }
        return None

    @after_model
    def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None
    ```

_`Example: Class`_

- 
    ```python
    from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
    from langchain.messages import AIMessage
    from langgraph.runtime import Runtime
    from typing import Any

    class MessageLimitMiddleware(AgentMiddleware):
        def __init__(self, max_messages: int = 50):
            super().__init__()
            self.max_messages = max_messages

        @hook_config(can_jump_to=["end"])
        def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
            if len(state["messages"]) >= self.max_messages:
                return {
                    "messages": [AIMessage("Conversation limit reached.")],
                    "jump_to": "end"
                }
            return None

        def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
            print(f"Model returned: {state['messages'][-1].content}")
            return None
    ```

#### 2.1.2. Wrap-style hooks

Intercept execution and control when the handler is called. Use for retries, caching, and transformation.

You decide if the handler is called zero times (short-circuit), once (normal flow), or multiple times (retry logic).

Available hooks are:

| Hook            | When it runs           |
|:----------------|:-----------------------|
| `wrap_model_call` | Around each model call |
| `wrap_tool_call`  | Around each tool call  |

_`Example: Decorator`_

- 
    ```python
    from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
    from typing import Callable

    class RetryMiddleware(AgentMiddleware):
        def __init__(self, max_retries: int = 3):
            super().__init__()
            self.max_retries = max_retries

        def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            for attempt in range(self.max_retries):
                try:
                    return handler(request)
                except Exception as e:
                    if attempt == self.max_retries - 1:
                        raise
                    print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")
    ```

_`Example: Class`_

- 
    ```python
    from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
    from typing import Callable

    class RetryMiddleware(AgentMiddleware):
        def __init__(self, max_retries: int = 3):
            super().__init__()
            self.max_retries = max_retries

        def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            for attempt in range(self.max_retries):
                try:
                    return handler(request)
                except Exception as e:
                    if attempt == self.max_retries - 1:
                        raise
                    print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")
    ```

### 2.2. State updates

Both node-style and wrap-style hooks can update agent state. The mechanism differs:
- __Node-style hooks__ (`before_agent`, `before_model`, `after_model`, `after_agent`):
    
    Return a dict directly. The dict is applied to the agent state using the graph’s reducers.
- __Wrap-style hooks__ (`wrap_model_call`, `wrap_tool_call`):
    
    For model calls, return `ExtendedModelResponse` with a `Command` to inject state updates alongside the model response. For tool calls, return a `Command` directly. Use these when you need to track or update state based on logic that runs during the model or tool call, such as summarization trigger points, usage metadata, or custom fields calculated from the request or response.

#### 2.2.1. Node-style hooks

Return a dict from a node-style hook to merge updates into agent state. The dict keys map to state fields.

_`Example:`_

- 
    ```python
    from langchain.agents.middleware import after_model, AgentState
    from langgraph.runtime import Runtime
    from typing import Any
    from typing_extensions import NotRequired

    class TrackingState(AgentState):
        model_call_count: NotRequired[int]

    @after_model(state_schema=TrackingState)
    def increment_after_model(
        state: TrackingState, runtime: Runtime
    ) -> dict[str, Any] | None:
        return {"model_call_count": state.get("model_call_count", 0) + 1}
    ```

#### 2.2.3. Wrap-style hooks

Returns a `ExtendedModelResponse` with a `Command` from `wrap_model_call` to inject state updates from the model call layer.

_`Example:`_

- 
    ```python
    from typing import Callable
    from langchain.agents.middleware import (
        wrap_model_call,
        ModelRequest,
        ModelResponse,
        AgentState,
        ExtendedModelResponse
    )
    from langgraph.types import Command
    from typing_extensions import NotRequired

    class UsageTrackingState(AgentState):
        """Agent state with token usage tracking."""

        last_model_call_tokens: NotRequired[int]

    @wrap_model_call(state_schema=UsageTrackingState)
    def track_usage(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ExtendedModelResponse:
        response = handler(request)
        return ExtendedModelResponse(
            model_response=response,
            command=Command(update={"last_model_call_tokens": 150}),
        )
    ```

The `Command` flows through the graph’s reducers, so updates are applied correctly and messages are additive rather than replacing existing state.
​
__Composition with multiple middleware__

When multiple middleware layers return `ExtendedModelResponse`, their commands compose:
- __Commands are applied through reducers__: Each `Command` becomes a separate state update. For messages, this means they are additive.
- __Outer wins on conflicts__: For non-reducer state fields, commands are applied inner-first, then outer. The outermost middleware’s value takes precedence on conflicting keys.
- __Retry-safe__: If the outer middleware implements logic that can result in multiple calls to `handler()` again (for example, retry logic), commands from earlier calls are discarded.

_`Example:`_

- 
    ```python
    from typing import Annotated, Callable
    from langchain.agents.middleware import (
        AgentMiddleware,
        AgentState,
        ExtendedModelResponse,
        ModelRequest,
        ModelResponse,
    )
    from langchain.messages import SystemMessage
    from langgraph.types import Command
    from typing_extensions import NotRequired

    def _last_wins(_a: str, b: str) -> str:
        """Reducer: last writer wins (outer overwrites inner)."""
        return b

    class CustomMiddlewareState(AgentState):
        """Agent state: trace_layer uses last-wins (outer wins), messages use additive reducer."""

        # Non-reducer field with last-wins: both middleware write; outermost value wins
        trace_layer: NotRequired[Annotated[str, _last_wins]]

    class OuterMiddleware(AgentMiddleware):
        def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ) -> ExtendedModelResponse:
            response = handler(request)
            return ExtendedModelResponse(
                model_response=response,
                command=Command(update={
                    "trace_layer": "outer",
                    "messages": [SystemMessage(content="[Outer ran]")],
                }),
            )

    class InnerMiddleware(AgentMiddleware):
        """Adds trace_layer and message. Outer adds to same keys; trace_layer: outer wins, messages: additive."""

        def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ):
            response = handler(request)
            return ExtendedModelResponse(
                model_response=response,
                command=Command(update={
                    "trace_layer": "inner",
                    "messages": [SystemMessage(content="[Inner ran]")],
                }),
            )
    ```

### 2.3. Create middleware

You can create middleware in two ways:
- __Decorator-based middleware__: Quick and simple for single-hook middleware. Use decorators to wrap individual functions.
- __Class-based middleware__: More powerful for complex middleware with multiple hooks or configuration.

#### 2.3.1. Decorator-based middleware

Quick and simple for single-hook middleware. Use decorators to wrap individual functions.

Available decorators are:
- __Node-style__:
    - `@before_agent` - Runs before agent starts (once per invocation)
    - `@before_model` - Runs before each model call
    - `@after_model` - Runs after each model response
    - `@after_agent` - Runs after agent completes (once per invocation)
- __Wrap-style__:
    - `@wrap_model_call` - Wraps each model call with custom logic
    - `@wrap_tool_call` - Wraps each tool call with custom logic
- __Convenience__:
    - `@dynamic_prompt` - Generates dynamic system prompts

Use decorators, when:
- Single hook needed
- No complex configuration
- Quick prototyping

_`Example:`_

- 
    ```python
    from langchain.agents.middleware import (
        before_model,
        wrap_model_call,
        AgentState,
        ModelRequest,
        ModelResponse,
    )
    from langchain.agents import create_agent
    from langgraph.runtime import Runtime
    from typing import Any, Callable

    @before_model
    def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"About to call model with {len(state['messages'])} messages")
        return None

    @wrap_model_call
    def retry_model(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        for attempt in range(3):
            try:
                return handler(request)
            except Exception as e:
                if attempt == 2:
                    raise
                print(f"Retry {attempt + 1}/3 after error: {e}")

    agent = create_agent(
        model="gpt-4.1",
        middleware=[log_before_model, retry_model],
        tools=[...],
    )
    ```

#### 2.3.3. Class-based middleware

More powerful for complex middleware with multiple hooks or configuration. Use classes when you need to define both sync and async implementations for the same hook, or when you want to combine multiple hooks in a single middleware.

Use classes, when:
- Defining both sync and async implementations for the same hook
- Multiple hooks needed in a single middleware
- Complex configuration required (e.g., configurable thresholds, custom models)
- Reuse across projects with init-time configuration

_`Example:`_

- 
    ```python
    from langchain.agents.middleware import (
        AgentMiddleware,
        AgentState,
        ModelRequest,
        ModelResponse,
    )
    from langgraph.runtime import Runtime
    from typing import Any, Callable

    class LoggingMiddleware(AgentMiddleware):
        def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
            print(f"About to call model with {len(state['messages'])} messages")
            return None

        def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
            print(f"Model returned: {state['messages'][-1].content}")
            return None

        async def abefore_model(
            self, state: AgentState, runtime: Runtime
        ) -> dict[str, Any] | None:
            # Async version of before_model
            return None

        async def aafter_model(
            self, state: AgentState, runtime: Runtime
        ) -> dict[str, Any] | None:
            # Async version of after_model
            print(f"Model returned: {state['messages'][-1].content}")
            return None


    agent = create_agent(
        model="gpt-4.1",
        middleware=[LoggingMiddleware()],
        tools=[...],
    )
    ```

### 2.4. Custom state schema

If your middleware needs to track state across hooks, middleware can extend the agent’s state with custom properties. This enables middleware to:
- __Track state across execution__: Maintain counters, flags, or other values that persist throughout the agent’s execution lifecycle
- __Share data between hooks__: Pass information from `before_model` to `after_model` or between different middleware instances
- __Implement cross-cutting concerns__: Add functionality like rate limiting, usage tracking, user context, or audit logging without modifying the core agent logic
- __Make conditional decisions__: Use accumulated state to determine whether to continue execution, jump to different nodes, or modify behavior dynamically

_`Example: Decorator`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.messages import HumanMessage
    from langchain.agents.middleware import AgentState, before_model, after_model
    from typing_extensions import NotRequired
    from typing import Any
    from langgraph.runtime import Runtime

    class CustomState(AgentState):
        model_call_count: NotRequired[int]
        user_id: NotRequired[str]

    @before_model(state_schema=CustomState, can_jump_to=["end"])
    def check_call_limit(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
        count = state.get("model_call_count", 0)
        if count > 10:
            return {"jump_to": "end"}
        return None

    @after_model(state_schema=CustomState)
    def increment_counter(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
        return {"model_call_count": state.get("model_call_count", 0) + 1}

    agent = create_agent(
        model="gpt-4.1",
        middleware=[check_call_limit, increment_counter],
        tools=[],
    )

    # Invoke with custom state
    result = agent.invoke({
        "messages": [HumanMessage("Hello")],
        "model_call_count": 0,
        "user_id": "user-123",
    })
    ```

_`Example: Class`_

- 
    ```python
    from langchain.agents import create_agent
    from langchain.messages import HumanMessage
    from langchain.agents.middleware import AgentState, AgentMiddleware
    from typing_extensions import NotRequired
    from typing import Any

    class CustomState(AgentState):
        model_call_count: NotRequired[int]
        user_id: NotRequired[str]

    class CallCounterMiddleware(AgentMiddleware[CustomState]):
        state_schema = CustomState

        def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
            count = state.get("model_call_count", 0)
            if count > 10:
                return {"jump_to": "end"}
            return None

        def after_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
            return {"model_call_count": state.get("model_call_count", 0) + 1}

    agent = create_agent(
        model="gpt-4.1",
        middleware=[CallCounterMiddleware()],
        tools=[],
    )

    # Invoke with custom state
    result = agent.invoke({
        "messages": [HumanMessage("Hello")],
        "model_call_count": 0,
        "user_id": "user-123",
    })
    ```

### 2.5. Execution order

When using multiple middleware, understand how they execute.

__Key rules__:
- `before_*` hooks: First to last
- `after_*` hooks: Last to first (reverse)
- `wrap_*` hooks: Nested (first middleware wraps all others)

__Execution flow__:
- _Before hooks run in order_:
    - `middleware1.before_agent()`
    - `middleware2.before_agent()`
    - `middleware3.before_agent()`
- _Agent loop starts_:
    - `middleware1.before_model()`
    - `middleware2.before_model()`
    - `middleware3.before_model()`
- _Wrap hooks nest like function calls_:
    - `middleware1.wrap_model_call()` → `middleware2.wrap_model_call()` → `middleware3.wrap_model_call()` → model
- _After hooks run in reverse order_:
    - `middleware3.after_model()`
    - `middleware2.after_model()`
    - `middleware1.after_model()`
- _Agent loop ends_:
    - `middleware3.after_agent()`
    - `middleware2.after_agent()`
    - `middleware1.after_agent()`

_`Example:`_

- 
    ```python
    agent = create_agent(
        model="gpt-4.1",
        middleware=[middleware1, middleware2, middleware3],
        tools=[...],
    )
    ```

### 2.6. Agent jumps

To exit early from middleware, return a dictionary with `jump_to`.

Available jump targets are:
- `'end'`: Jump to the end of the agent execution (or the first `after_agent` hook)
- `'tools'`: Jump to the tools node
- `'model'`: Jump to the model node (or the first `before_model` hook)

_`Example: Decorator`_

- 
    ```python
    from langchain.agents.middleware import after_model, hook_config, AgentState
    from langchain.messages import AIMessage
    from langgraph.runtime import Runtime
    from typing import Any

    @after_model
    @hook_config(can_jump_to=["end"])
    def check_for_blocked(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        if "BLOCKED" in last_message.content:
            return {
                "messages": [AIMessage("I cannot respond to that request.")],
                "jump_to": "end"
            }
        return None
    ```

_`Example: Class`_

- 
    ```python
    from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState
    from langchain.messages import AIMessage
    from langgraph.runtime import Runtime
    from typing import Any

    class BlockedContentMiddleware(AgentMiddleware):
        @hook_config(can_jump_to=["end"])
        def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
            last_message = state["messages"][-1]
            if "BLOCKED" in last_message.content:
                return {
                    "messages": [AIMessage("I cannot respond to that request.")],
                    "jump_to": "end"
                }
            return None
    ```

### 2.7. Best practices

1. Keep middleware focused - each should do one thing well
2. Handle errors gracefully - don’t let middleware errors crash the agent
3. __Use appropriate hook types__:
    - Node-style for sequential logic (logging, validation)
    - Wrap-style for control flow (retry, fallback, caching)
4. Clearly document any custom state properties
5. Unit test middleware independently before integrating
6. Consider execution order - place critical middleware first in the list
7. Use built-in middleware when possible

Check <a href="https://docs.langchain.com/oss/python/langchain/middleware/custom#examples">`Examples`</a> for more details.

